In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import matplotlib.pyplot as plt

In [9]:
# Load heat capacity dataset
df = pd.read_csv("cleaned_cp_dataset.csv")

print("Dataset loaded successfully!")
print("Original shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Dataset loaded successfully!
Original shape: (4564, 3)
Columns:
['formula', 'T', 'Cp']


In [10]:
# Remove extra spaces from column names
df.columns = df.columns.astype(str).str.strip()

print("Cleaned columns:")
print(df.columns.tolist())

Cleaned columns:
['formula', 'T', 'Cp']


In [11]:
required_columns = ["formula", "Temperature", "Heat_Capacity"]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required columns are available.")

Missing columns: ['Temperature', 'Heat_Capacity']


In [14]:
# S26 - Data Preparation

# Clean column names
df.columns = df.columns.astype(str).str.strip()

print("Available columns:")
print(df.columns.tolist())

# Correct columns in our dataset
formula_column = "formula"
temperature_column = "T"
heat_capacity_column = "Cp"

# Check columns
required_columns = [
    formula_column,
    temperature_column,
    heat_capacity_column
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(f"Missing columns: {missing_columns}")

# Keep required columns
df_model = df[
    [formula_column, temperature_column, heat_capacity_column]
].copy()

# Remove missing values
df_model = df_model.dropna()

# Convert T and Cp to numeric
df_model[temperature_column] = pd.to_numeric(
    df_model[temperature_column],
    errors="coerce"
)

df_model[heat_capacity_column] = pd.to_numeric(
    df_model[heat_capacity_column],
    errors="coerce"
)

# Remove invalid values
df_model = df_model.dropna()

# Input and target
X = df_model[[temperature_column]]
y = df_model[heat_capacity_column]

print("\nData preparation completed successfully!")
print("Dataset shape:", df_model.shape)
print("Input columns:", X.columns.tolist())
print("Target column:", y.name)

Available columns:
['formula', 'T', 'Cp']

Data preparation completed successfully!
Dataset shape: (4564, 3)
Input columns: ['T']
Target column: Cp


In [15]:
# S26 - Train Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Number of features:", X_train.shape[1])

Training samples: 3651
Testing samples: 913
Number of features: 1


In [16]:
# S26 - Lasso, ElasticNet & Sparsity

import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# 1. Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)


# 2. Ridge Regression
# Best alpha obtained from S25
best_alpha = 0.010

ridge_model = Ridge(alpha=best_alpha)
ridge_model.fit(X_train, y_train)

ridge_predictions = ridge_model.predict(X_test)


# 3. Lasso Regression
lasso_model = Lasso(alpha=1.0, random_state=42)
lasso_model.fit(X_train, y_train)

lasso_predictions = lasso_model.predict(X_test)


# Find zeroed features
zeroed_features = X_train.columns[lasso_model.coef_ == 0]

print("Lasso forced", len(zeroed_features),
      "features to exactly 0.")

print("Zeroed features:", list(zeroed_features))


# 4. ElasticNet Regression
elastic_model = ElasticNet(
    alpha=1.0,
    l1_ratio=0.5,
    random_state=42
)

elastic_model.fit(X_train, y_train)

elastic_predictions = elastic_model.predict(X_test)


# 5. Evaluation function
def evaluate_model(y_true, y_pred):
    return {
        "R²": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred)
    }


# 6. Four-model comparison
results = pd.DataFrame({
    "Unregularized LR": evaluate_model(
        y_test,
        linear_predictions
    ),

    "Ridge (L2)": evaluate_model(
        y_test,
        ridge_predictions
    ),

    "Lasso (L1)": evaluate_model(
        y_test,
        lasso_predictions
    ),

    "ElasticNet": evaluate_model(
        y_test,
        elastic_predictions
    )
}).T


print("\n--- 4-Model Evaluation Metrics ---")
display(results)

Lasso forced 0 features to exactly 0.
Zeroed features: []

--- 4-Model Evaluation Metrics ---


,R²,RMSE,MAE
Unregularized LR,0.015058,68.234733,49.495365
Ridge (L2),0.015058,68.234733,49.495365
Lasso (L1),0.015057,68.234794,49.495366
ElasticNet,0.015058,68.234764,49.495366


In [17]:
# Save the deliverable

results.to_csv(
    "regularization_comparison_s26.csv"
)

print("✅ 4-model comparison table successfully saved.")

✅ 4-model comparison table successfully saved.
